# 01 — Data Exploration
**Benin Least-Cost Electrification Analysis**

This notebook loads and explores the **GIS-enriched settlement dataset** produced
by Notebook 00 (`settlements_gis_enriched.geojson`), saved to `data/processed/`.
It combines:

- **VIDA / DRE Atlas** — 17,205 settlements with demand, wealth, building mix, grid distances
- **Oxford MAP raster** — motorized travel time to nearest city (`TravelHours`)
- **NASA POWER / GlobalSolarAtlas** — GHI solar irradiance
- **GlobalWindAtlas** — wind speed and capacity factor
- **SRTM DEM** — elevation and slope
- **HydroSHEDS** — hydropower potential
- **VIIRS 2020** — night-time lights
- **MODIS MCD12Q1 2022** — land cover and productive use factor

> ⚠ **Run Notebook 00 first** to generate `data/processed/settlements_gis_enriched.geojson`.
> This notebook will fall back to raw VIDA data if the enriched file is not yet available.

**Sections:**
1. Load enriched data
2. Dataset overview & data quality
3. VIDA demand & wealth distributions
4. GIS resource layers (solar, wind, hydro, terrain)
5. Accessibility & diesel price
6. Grid infrastructure
7. Land cover & social services
8. Correlation matrix — resource layers
9. Transmission lines
10. Exploratory map


In [ ]:
import sys
sys.path.append('..')

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
import warnings

try:
    import folium
    HAS_FOLIUM = True
except ImportError:
    HAS_FOLIUM = False
    print('folium not installed — map cell will be skipped')

BASE_DIR    = Path('..').resolve()
OUTPUT_DIR  = BASE_DIR / 'data' / 'outputs' / 'maps'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='muted')
print('Libraries loaded ✓')


## 1. Load Enriched Data

Loads the GIS-enriched settlements file produced by NB00.
If NB00 has not been run yet, falls back to the raw VIDA file — some columns (slope, GHI, diesel price,
hydro) will be missing and plots will degrade gracefully.

The three `print()` calls that follow give:
- **Column inventory** grouped by theme (VIDA originals, GIS-enriched, geometry)
- **Missing value count** for each resource column — anything > 5 % is flagged
- **Basic shape/CRS check** to confirm the file loaded correctly


In [ ]:
# Load GIS-enriched settlements (output of Notebook 00)
# Falls back to raw VIDA file if enriched version not yet generated

PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
RAW_DIR       = BASE_DIR / 'data' / 'raw'

# Try enriched first, then raw fallback
enriched_candidates = sorted(
    list(PROCESSED_DIR.glob('settlements_gis_enriched*.geojson')),
    reverse=True   # newest timestamp first
)
if enriched_candidates:
    SETTLEMENTS_PATH = enriched_candidates[0]
    DATA_SOURCE = 'GIS-enriched (Notebook 00)'
else:
    SETTLEMENTS_PATH = RAW_DIR / 'Benin_settlement_properties.geojson'
    DATA_SOURCE = 'Raw VIDA (Notebook 00 not yet run)'
    warnings.warn('Enriched file not found — run Notebook 00 first for full analysis')

TRANSMISSION_PATH = RAW_DIR / 'Benin_existing_transmission_lines_2017.geojson'

print(f'Data source      : {DATA_SOURCE}')
print(f'Settlements file : {SETTLEMENTS_PATH.name}')
print(f'Transmission file: {TRANSMISSION_PATH.name}  |  exists: {TRANSMISSION_PATH.exists()}')

gdf = gpd.read_file(SETTLEMENTS_PATH)
gdf['lon'] = gdf.geometry.centroid.x
gdf['lat'] = gdf.geometry.centroid.y

lines = gpd.read_file(TRANSMISSION_PATH) if TRANSMISSION_PATH.exists() else None
if lines is not None:
    print(f'  Lines loaded: {len(lines)} features  |  columns: {list(lines.columns)}')

print(f'\nSettlements : {len(gdf):,}')
print(f'Columns     : {len(gdf.columns)}')
print(f'CRS         : {gdf.crs}')


In [ ]:
# ── Dataset overview ──────────────────────────────────────────────────────────
print('=== COLUMN INVENTORY ===')

# Group columns by category
col_groups = {
    'VIDA core'       : ['identifier','village_name','population','num_buildings',
                         'num_connections','energy_demand','avg_connection_energy_demand',
                         'mean_rwi','Medium_and_large_buildings_pc'],
    'Solar / Wind'    : ['GHI','WindVel','WindCF'],
    'Terrain'         : ['Slope','Elevation'],
    'Hydro'           : ['HydropowerDist','Hydropower','HydroHead','HydroDischarge'],
    'Accessibility'   : ['travel_time_raster_min','TravelHours','DieselPrice'],
    'Grid'            : ['GridDistKm','GridProximity','DistPlannedLine',
                         'DistSubstation','CityDistKm'],
    'MG inputs'       : ['dist_river_km','dist_road_km'],
    'Night lights'    : ['NightLights'],
    'Land cover'      : ['LandCover','LandCoverLabel','ProductiveUseFactor'],
}

for group, cols in col_groups.items():
    present = [c for c in cols if c in gdf.columns]
    missing = [c for c in cols if c not in gdf.columns]
    print(f'  {group:<18}: {len(present)}/{len(cols)} present'
          + (f'  ← missing: {missing}' if missing else '  ✓'))

print(f'\nBounding box: '
      f'lon [{gdf.lon.min():.2f}, {gdf.lon.max():.2f}]  '
      f'lat [{gdf.lat.min():.2f}, {gdf.lat.max():.2f}]')


In [ ]:
# ── Data quality — missing values ─────────────────────────────────────────────
resource_cols = [
    'GHI','WindVel','WindCF','Slope','Elevation',
    'HydropowerDist','Hydropower',
    'travel_time_raster_min','TravelHours','DieselPrice',
    'GridDistKm','DistPlannedLine','CityDistKm',
    'dist_river_km','dist_road_km',
    'NightLights','LandCover','ProductiveUseFactor',
]
present_cols = [c for c in resource_cols if c in gdf.columns]

print(f'=== DATA QUALITY — GIS RESOURCE LAYERS ({len(present_cols)} cols) ===')
print(f'  {"Column":<28} {"Valid":>8}  {"Missing":>8}  {"Min":>10}  {"Mean":>10}  {"Max":>10}')
print('  ' + '─'*78)
for col in present_cols:
    s       = pd.to_numeric(gdf[col], errors='coerce')
    n_valid = s.notna().sum()
    n_miss  = s.isna().sum()
    if n_valid > 0:
        print(f'  {col:<28} {n_valid:>8,}  {n_miss:>8,}  '
              f'{s.min():>10.3f}  {s.mean():>10.3f}  {s.max():>10.3f}')
    else:
        print(f'  {col:<28} {"NOT IN DATA":>28}')


## 2. VIDA Demand & Wealth Distributions

Six-panel figure exploring the core VIDA variables used in demand estimation and tier assignment.

**What to look for:**
- `energy_demand` distribution should be right-skewed — most settlements have < 5 kWh/day (small villages)
- `mean_rwi` should centre near 0 (by design of the RWI scale) with a longer left tail (poor rural areas)
- `num_connections` bimodal distribution is expected — many small (<20 HH) and a few large (>200 HH) settlements
- Scatter of `energy_demand` vs `num_connections` should show near-linear relationship (VIDA uses building counts to compute demand)

**Expected finding for Benin:** median settlement demand ≈ 2,500–4,000 kWh/yr (equivalent to ~35–55 households at Tier-2).


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('VIDA Core Variables — Benin Settlements (n=17,205)', fontsize=13, fontweight='bold')

# Population
axes[0,0].hist(gdf['population'].clip(upper=gdf['population'].quantile(0.98)),
               bins=50, color='#2196F3', edgecolor='white', linewidth=0.3)
axes[0,0].set_title('Population per settlement')
axes[0,0].set_xlabel('Population')

# Energy demand (kWh/day)
demand_col = 'energy_demand' if 'energy_demand' in gdf.columns else None
if demand_col:
    axes[0,1].hist(gdf[demand_col].clip(upper=gdf[demand_col].quantile(0.98)),
                   bins=50, color='#FF9800', edgecolor='white', linewidth=0.3)
    axes[0,1].set_title('Energy demand (kWh/day)')
    axes[0,1].set_xlabel('kWh/day')

# Num connections (buildings)
axes[0,2].hist(gdf['num_connections'].clip(upper=gdf['num_connections'].quantile(0.98)),
               bins=50, color='#4CAF50', edgecolor='white', linewidth=0.3)
axes[0,2].set_title('Connections (buildings)')
axes[0,2].set_xlabel('Number of connections')

# Mean RWI
axes[1,0].hist(gdf['mean_rwi'].dropna(), bins=50, color='#9C27B0',
               edgecolor='white', linewidth=0.3)
axes[1,0].set_title('Mean RWI (Relative Wealth Index)')
axes[1,0].axvline(0, color='red', linestyle='--', lw=1, label='RWI=0')
axes[1,0].axvline(0.5, color='orange', linestyle='--', lw=1, label='High threshold')
axes[1,0].legend(fontsize=8)
axes[1,0].set_xlabel('RWI')

# Medium+large buildings %
axes[1,1].hist(gdf['Medium_and_large_buildings_pc'].dropna(), bins=50,
               color='#795548', edgecolor='white', linewidth=0.3)
axes[1,1].set_title('Medium+Large Buildings (%)')
axes[1,1].axvline(0.10, color='orange', linestyle='--', lw=1, label='Medium threshold')
axes[1,1].axvline(0.30, color='red', linestyle='--', lw=1, label='High threshold')
axes[1,1].legend(fontsize=8)

# Avg connection energy demand
avg_col = 'avg_connection_energy_demand'
if avg_col in gdf.columns:
    axes[1,2].hist(gdf[avg_col].clip(upper=gdf[avg_col].quantile(0.98)),
                   bins=50, color='#F44336', edgecolor='white', linewidth=0.3)
    axes[1,2].set_title('Avg demand per connection (kWh/day)')
    axes[1,2].set_xlabel('kWh/day/connection')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_vida_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary stats
print('=== VIDA DEMAND SUMMARY ===')
for col in ['population','num_connections','energy_demand','avg_connection_energy_demand','mean_rwi']:
    if col in gdf.columns:
        s = pd.to_numeric(gdf[col], errors='coerce')
        print(f'  {col:<38}: median={s.median():.2f}  mean={s.mean():.2f}  max={s.max():.2f}')


## 3. GIS Resource Layers — Solar, Wind, Hydro, Terrain

Resource layers extracted in **Notebook 06** from global raster datasets.


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('GIS Resource Layers — Benin Settlements', fontsize=13, fontweight='bold')

panels = [
    ('GHI',          'GHI (kWh/m²/day)',         '#FFC107', (3.5, 7.0),  'Solar irradiance'),
    ('WindVel',      'Wind speed (m/s)',           '#03A9F4', (0,   8),   'Wind speed'),
    ('WindCF',       'Wind capacity factor',       '#00BCD4', (0,   0.6), 'Wind CF'),
    ('Slope',        'Slope (degrees)',             '#8BC34A', (0,  20),   'Terrain slope'),
    ('Elevation',    'Elevation (m)',               '#607D8B', (0, 800),   'Elevation'),
    ('Hydropower',   'Hydro potential (kW)',        '#1565C0', None,       'Hydro potential'),
    ('HydropowerDist','Dist to hydro site (km)',   '#42A5F5', (0,  50),   'Dist to hydro'),
    ('NightLights',  'VIIRS night lights (DN)',     '#FF7043', None,       'Night lights'),
]

for ax, (col, xlabel, color, xlim, title) in zip(axes.flat, panels):
    if col in gdf.columns:
        s = pd.to_numeric(gdf[col], errors='coerce').dropna()
        if xlim:
            s = s.clip(*xlim)
        ax.hist(s, bins=50, color=color, edgecolor='white', linewidth=0.3)
        ax.set_title(title, fontsize=10)
        ax.set_xlabel(xlabel, fontsize=9)
        ax.tick_params(labelsize=8)
    else:
        ax.text(0.5, 0.5, f'{col}\nnot available', ha='center', va='center',
                transform=ax.transAxes, fontsize=9, color='gray')
        ax.set_title(title, fontsize=10)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_resource_layers.png', dpi=150, bbox_inches='tight')
plt.show()

print('=== RESOURCE LAYER SUMMARY ===')
for col, label, *_ in panels:
    if col in gdf.columns:
        s = pd.to_numeric(gdf[col], errors='coerce')
        print(f'  {label:<32}: median={s.median():.3f}  '
              f'mean={s.mean():.3f}  max={s.max():.3f}  '
              f'valid={s.notna().sum():,}')


## 4. Accessibility & Diesel Price

Travel time extracted from the **Oxford MAP accessibility raster** (Weiss et al. 2020).
Diesel price computed using the **OnSSET transport logistics model**
(`p_d + 2 × p_d × C_truck × TravelHours / V_truck`).


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Accessibility & Diesel Price — Benin Settlements', fontsize=12, fontweight='bold')

# Travel time (minutes)
if 'travel_time_raster_min' in gdf.columns:
    s = pd.to_numeric(gdf['travel_time_raster_min'], errors='coerce')
    axes[0].hist(s.clip(upper=s.quantile(0.98)), bins=50, color='#26A69A',
                 edgecolor='white', linewidth=0.3)
    axes[0].set_title('Travel time to nearest city')
    axes[0].set_xlabel('Minutes (MAP raster)')
    axes[0].axvline(s.median(), color='red', lw=1.5, linestyle='--',
                    label=f'Median {s.median():.0f} min')
    axes[0].legend(fontsize=8)

# TravelHours
if 'TravelHours' in gdf.columns:
    t = pd.to_numeric(gdf['TravelHours'], errors='coerce')
    axes[1].hist(t.clip(upper=t.quantile(0.98)), bins=50, color='#42A5F5',
                 edgecolor='white', linewidth=0.3)
    axes[1].set_title('Travel hours')
    axes[1].set_xlabel('Hours (TravelHours)')
    axes[1].axvline(t.median(), color='red', lw=1.5, linestyle='--',
                    label=f'Median {t.median():.2f} hr')
    axes[1].legend(fontsize=8)

# Diesel delivered price
if 'DieselPrice' in gdf.columns:
    d = pd.to_numeric(gdf['DieselPrice'], errors='coerce')
    axes[2].hist(d, bins=50, color='#EF5350', edgecolor='white', linewidth=0.3)
    axes[2].set_title('Diesel delivered price')
    axes[2].set_xlabel('USD/litre (OnSSET logistics model)')
    axes[2].axvline(d.median(), color='darkred', lw=1.5, linestyle='--',
                    label=f'Median ${d.median():.4f}/L')
    axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_accessibility.png', dpi=150, bbox_inches='tight')
plt.show()

print('=== ACCESSIBILITY SUMMARY ===')
for col, label in [('travel_time_raster_min','Travel time (min)'),
                    ('TravelHours','TravelHours (hr)'),
                    ('DieselPrice','Diesel price (USD/L)')]:
    if col in gdf.columns:
        s = pd.to_numeric(gdf[col], errors='coerce')
        print(f'  {label:<24}: median={s.median():.4f}  '
              f'min={s.min():.4f}  max={s.max():.4f}')


## 5. Grid Infrastructure

Grid distances from **VIDA / DRE Atlas** (World Bank energydata.info).
`GridDistKm` = distance to existing transmission line — the primary LCOE input.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Grid Infrastructure — Benin Settlements', fontsize=12, fontweight='bold')

# GridDistKm distribution
if 'GridDistKm' in gdf.columns:
    g = pd.to_numeric(gdf['GridDistKm'], errors='coerce')
    axes[0].hist(g.clip(upper=100), bins=60, color='#4CAF50',
                 edgecolor='white', linewidth=0.3)
    axes[0].axvline(25, color='red', lw=1.5, linestyle='--', label='25 km threshold')
    axes[0].set_title('Distance to existing grid')
    axes[0].set_xlabel('km (clipped at 100)')
    axes[0].legend(fontsize=8)

    # Threshold summary
    print('=== GRID PROXIMITY SUMMARY ===')
    for t in [5, 10, 15, 25, 50]:
        n   = (g <= t).sum()
        pct = n / len(g) * 100
        print(f'  ≤ {t:3d} km : {n:5,} ({pct:.1f}%)')
    print(f'  >  25 km : {(g > 25).sum():5,} ({(g > 25).mean()*100:.1f}%) — SHS/Mini-Grid candidates')

# GridProximity buckets
if 'GridProximity' in gdf.columns:
    counts = gdf['GridProximity'].value_counts().sort_index()
    axes[1].bar(counts.index.astype(str), counts.values,
                color=['#1B5E20','#388E3C','#FFA000','#E53935'])
    axes[1].set_title('Grid proximity buckets')
    axes[1].set_xlabel('Distance band')
    axes[1].set_ylabel('Settlements')
    for i, v in enumerate(counts.values):
        axes[1].text(i, v + 50, f'{v:,}', ha='center', fontsize=8)

# CityDistKm vs GridDistKm scatter
if 'CityDistKm' in gdf.columns and 'GridDistKm' in gdf.columns:
    sample = gdf.sample(min(3000, len(gdf)), random_state=42)
    axes[2].scatter(
        pd.to_numeric(sample['GridDistKm'], errors='coerce').clip(upper=100),
        pd.to_numeric(sample['CityDistKm'], errors='coerce').clip(upper=60),
        alpha=0.15, s=4, color='steelblue', rasterized=True
    )
    axes[2].set_title('Grid dist vs city dist (sample)')
    axes[2].set_xlabel('GridDistKm (km)')
    axes[2].set_ylabel('CityDistKm (km)')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_grid_infrastructure.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Land Cover & Social Services

**Land cover** (MODIS IGBP classes) is used in two places in the model:
1. **OnSSET terrain penalty** for grid extension — woodland/forest = higher construction cost
2. **Productive use factor** — cropland and urban classes receive a demand multiplier (×1.25 or ×1.40)

**Social services** (`has_health_facility`, `has_education_facility`) contribute +1 point to the MTF
tier scoring system in `mtf_tiers.py`. They are also potential anchor loads for mini-grid viability
(though not individually modelled — folded into the productive use factor).

**Expected finding:** ~15–25 % of settlements have a health or education facility. These tend to cluster
near existing grid lines (electrified towns).


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Land Cover & Social Services — Benin Settlements', fontsize=12, fontweight='bold')

# Land cover distribution
if 'LandCoverLabel' in gdf.columns:
    lc_counts = gdf['LandCoverLabel'].value_counts().head(8)
    axes[0].barh(lc_counts.index, lc_counts.values, color='#66BB6A')
    axes[0].set_title('Land cover (MODIS 2022)')
    axes[0].set_xlabel('Settlements')
    for i, v in enumerate(lc_counts.values):
        axes[0].text(v + 10, i, f'{v:,}', va='center', fontsize=8)

# Productive use factor
if 'ProductiveUseFactor' in gdf.columns:
    puf = gdf['ProductiveUseFactor'].value_counts().sort_index()
    axes[1].bar(puf.index.astype(str), puf.values,
                color=['#AED581','#FFA726','#EF5350'])
    axes[1].set_title('Productive Use Factor')
    axes[1].set_xlabel('Factor (×baseline demand)')
    axes[1].set_ylabel('Settlements')
    for i, (k, v) in enumerate(puf.items()):
        axes[1].text(i, v + 30, f'{v:,}\n({v/len(gdf)*100:.1f}%)',
                     ha='center', fontsize=8)

# Social services
social_cols = ['has_health_facility','has_education_facility','has_nightlight']
present_social = [c for c in social_cols if c in gdf.columns]
if present_social:
    vals  = [gdf[c].sum() for c in present_social]
    pcts  = [v/len(gdf)*100 for v in vals]
    labels = ['Health\nfacility', 'Education\nfacility', 'Night\nlight']
    bars = axes[2].bar(labels[:len(present_social)], vals,
                       color=['#EC407A','#AB47BC','#26C6DA'])
    axes[2].set_title('Social services & night light')
    axes[2].set_ylabel('Settlements')
    for bar, pct in zip(bars, pcts):
        axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                     f'{pct:.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_landcover_social.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
## 7. Correlation Matrix — Resource Layers

# Numeric resource columns available in the enriched dataset
corr_cols = [
    c for c in [
        'energy_demand','mean_rwi','Medium_and_large_buildings_pc',
        'GHI','WindVel','WindCF','Slope','Elevation',
        'Hydropower','HydropowerDist',
        'TravelHours','DieselPrice',
        'GridDistKm','CityDistKm',
        'dist_river_km','dist_road_km',
        'NightLights','ProductiveUseFactor',
    ] if c in gdf.columns
]

corr_df = gdf[corr_cols].apply(pd.to_numeric, errors='coerce').corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_df, dtype=bool), k=1)
sns.heatmap(
    corr_df, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
    center=0, vmin=-1, vmax=1, linewidths=0.5,
    annot_kws={'size': 7}, ax=ax
)
ax.set_title('Pearson Correlation — GIS Resource Layers\nBenin Settlements',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Top correlations with energy_demand
if 'energy_demand' in corr_df.columns:
    print('=== TOP CORRELATIONS WITH energy_demand ===')
    top = corr_df['energy_demand'].drop('energy_demand').abs().sort_values(ascending=False).head(8)
    for col, val in top.items():
        direction = '(+)' if corr_df.loc[col,'energy_demand'] > 0 else '(-)'
        print(f'  {col:<30}: r = {corr_df.loc[col,"energy_demand"]:+.3f}  {direction}')


## 8. Transmission Lines

Prints key attributes of the SBEE transmission line file. The `Situation` column distinguishes:
- **Existing** lines → used for Option B grid routing in NB03 (nearest electrified settlement)
- **Planned** lines → used for grid priority bonus (+25 % score if settlement is < 10 km from a planned line)

Check that both categories are present. If `Situation` only contains one value, the planned-line bonus
will not fire correctly in `technology_selector.py`.

**Expected voltage levels in Benin:** 63 kV (HV backbone), 33 kV (MV rural), 20 kV (peri-urban).


In [ ]:
if lines is not None:
    print('=== TRANSMISSION LINES ===')
    show_cols = [c for c in ['Name','Voltage_KV','Situation','km','Country_1'] if c in lines.columns]
    print(lines[show_cols].to_string())
    exist_mask = lines['Situation'].str.lower().str.contains('exist', na=False)
    print(f'\nExisting : {exist_mask.sum()}')
    print(f'Planned  : {(~exist_mask).sum()}')
else:
    print('Transmission lines file not found — skipping')


## 9. Exploratory Map

Settlements coloured by **grid distance** (blue=<10km, orange=10–25km, green=>25km).


In [ ]:
if not HAS_FOLIUM:
    print('folium not installed — pip install folium to enable map')
else:
    center = [gdf.lat.mean(), gdf.lon.mean()]
    m = folium.Map(location=center, zoom_start=7, tiles='CartoDB positron')

    dist_col = 'GridDistKm' if 'GridDistKm' in gdf.columns \
               else 'distance_to_existing_transmission_lines'

    for _, row in gdf.sample(min(2500, len(gdf)), random_state=42).iterrows():
        dist = row.get(dist_col, None)
        if pd.isna(dist):          color = 'gray'
        elif dist <= 10:           color = '#1565C0'
        elif dist <= 25:           color = '#FF8F00'
        else:                      color = '#2E7D32'

        tt   = row.get('TravelHours', '?')
        ghi  = row.get('GHI', '?')
        name = row.get('village_name', '?')

        folium.CircleMarker(
            location=[row.lat, row.lon],
            radius=3, color=color, fill=True,
            fill_color=color, fill_opacity=0.6,
            popup=(f"{name}<br>GridDist={dist:.1f}km<br>"
                   f"TravelHours={tt:.2f}hr<br>GHI={ghi:.2f}" if not pd.isna(dist) else name)
        ).add_to(m)

    if lines is not None:
        for _, row in lines.iterrows():
            try:
                coords = [(y, x) for x, y in row.geometry.coords]
                color  = 'red' if 'exist' in str(row.get('Situation','')).lower() else 'orange'
                folium.PolyLine(coords, color=color, weight=2.5, opacity=0.8,
                                tooltip=f"{row.get('Name','')} {row.get('Voltage_KV','')}kV"
                                ).add_to(m)
            except:
                pass

    map_path = OUTPUT_DIR / 'eda_map.html'
    m.save(str(map_path))
    print(f'Map saved → {map_path}')
    display(m)
